저장된 training_data_clean_dedup_balanced 데이터 정합성 테스트

In [3]:
#split후 논리검증
from sklearn.model_selection import train_test_split
import pandas as pd

PATH = "/content/drive/MyDrive/medical/Capstone/training_data_clean_dedup_balanced.csv"

# text_A만 모으기 (가벼움)
A_set = set()
for chunk in pd.read_csv(PATH, chunksize=50000, usecols=["text_A"]):
    A_set.update(chunk["text_A"].dropna().unique())

A_list = list(A_set)
train_A, valid_A = train_test_split(A_list, test_size=0.1, random_state=42)

train_A = set(train_A)
valid_A = set(valid_A)

# split 자체가 잘 됐는지
print("text_A overlap:", len(train_A & valid_A))  # 0이어야 함


text_A overlap: 0


In [5]:
#라벨 논리 검증
import pandas as pd

PATH = "/content/drive/MyDrive/medical/Capstone/training_data_clean_dedup_balanced.csv"

samples = []
for chunk in pd.read_csv(PATH, chunksize=50000):
    samples.append(chunk.sample(n=min(200, len(chunk)), random_state=42))
    if len(samples) >= 10:  # 약 2000개
        break

sdf = pd.concat(samples, ignore_index=True)

a = sdf["text_A"].astype(str).str.lower()
b = sdf["text_B"].astype(str).str.lower()

contains = pd.Series(
    [(bb in aa) for aa, bb in zip(a, b)]
)

print("pos contains rate:", contains[sdf["label"]==1].mean())
print("neg NOT-contains rate:", (~contains[sdf["label"]==0]).mean())

pos contains rate: 1.0
neg NOT-contains rate: 1.0


In [1]:
#(text_A,text_B)가 중복으로 다시 나오나 확인
import pandas as pd
import hashlib

PATH = "/content/drive/MyDrive/medical/Capstone/training_data_clean_dedup_balanced.csv"

seen_hashes = set()
dup_count = 0
MAX_SEEN = 3_000_000  # 안전 상한 (넘으면 중단)

def pair_hash(a, b):
    return hashlib.md5((a + "||" + b).encode("utf-8")).hexdigest()

for chunk in pd.read_csv(PATH, chunksize=50000):
    for a, b in zip(chunk["text_A"], chunk["text_B"]):
        h = pair_hash(a, b)
        if h in seen_hashes:
            dup_count += 1
            break
        seen_hashes.add(h)

        if len(seen_hashes) > MAX_SEEN:
            print("Reached hash memory limit, stopping early (safe)")
            break

    if dup_count > 0:
        break

print("duplicate pairs found:", dup_count)


Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
Reached hash memory limit, stopping early (safe)
duplicate pairs found: 0


In [1]:
import pandas as pd

PATH = "/content/drive/MyDrive/medical/Capstone/training_data_clean_dedup_balanced.csv"

chunk = pd.read_csv(PATH, nrows=1000)
print(chunk.columns)
print(chunk["label"].value_counts())


Index(['text_A', 'text_B', 'label'], dtype='object')
label
1    508
0    492
Name: count, dtype: int64


부정데이터 추가 생성후 한 파일로 저장

In [26]:
final_df.to_csv(
    "/content/drive/MyDrive/medical/Capstone/training_data_clean_dedup_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)


In [24]:
neg_df = pd.DataFrame(neg_samples)

final_df = pd.concat([pos_df, neg_df], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df["label"].value_counts()


,count
label,
1,1861811
0,1861811


In [23]:
import random
from tqdm import tqdm

neg_samples = []

with tqdm(total=target_neg, desc="Generating negatives") as pbar:
    while len(neg_samples) < target_neg:
        text_A = random.choice(criteria_pool)
        text_B = random.choice(term_pool)

        # ❌ 실제로 포함되면 negative 아님
        if text_B.lower() in text_A.lower():
            continue

        pair = (text_A, text_B)

        # ❌ 중복 방지
        if pair in seen_pairs:
            continue

        neg_samples.append({
            "text_A": text_A,
            "text_B": text_B,
            "label": 0
        })
        seen_pairs.add(pair)
        pbar.update(1)


Generating negatives: 100%|██████████| 1861811/1861811 [00:28<00:00, 64546.65it/s]


In [22]:
target_neg = len(pos_df)


In [21]:
seen_pairs = set(zip(pos_df["text_A"], pos_df["text_B"]))


In [20]:
term_pool = pos_df["text_B"].unique().tolist()


In [19]:
criteria_pool = pos_df["text_A"].unique().tolist()


In [18]:
pos_df = df[df["label"] == 1].copy()
print("positive:", len(pos_df))


positive: 1861811


데이터 합쳐서 중복 제거

In [17]:
df["label"].value_counts()


,count
label,
1,1861811
0,862998


In [16]:
import pandas as pd
import glob

files = glob.glob("/content/drive/MyDrive/medical/Capstone/final_training_data_regex_part*.csv")

df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print("Before dedup:", len(df))
df = df.drop_duplicates(subset=["text_A", "text_B"])
print("After dedup:", len(df))


Before dedup: 2735963
After dedup: 2724809


데이터 확인

In [15]:
print("\n===== [5] (text_A, text_B) 쌍 중복 확인 =====")

import pandas as pd

BASE_PATH = "/content/drive/MyDrive/medical/Capstone"

FILES = {
    1: f"{BASE_PATH}/final_training_data_regex_part1.csv",
    2: f"{BASE_PATH}/final_training_data_regex_part2.csv",
    3: f"{BASE_PATH}/final_training_data_regex_part3.csv",
    4: f"{BASE_PATH}/final_training_data_regex_part4.csv",
}

# 1️⃣ 모든 shard 로드
dfs = {}
for i, path in FILES.items():
    dfs[i] = pd.read_csv(path)
    print(f"part{i} loaded, rows={len(dfs[i])}")

# 2️⃣ (text_A, text_B) 쌍 중복 확인
print("\n===== (text_A, text_B) 쌍 중복 확인 =====")

for i in range(1, 4):
    pairs_i = set(zip(dfs[i]["text_A"], dfs[i]["text_B"]))
    pairs_j = set(zip(dfs[i+1]["text_A"], dfs[i+1]["text_B"]))
    overlap_pairs = pairs_i.intersection(pairs_j)
    print(f"part{i} ↔ part{i+1} pair overlap: {len(overlap_pairs)}")




===== [5] (text_A, text_B) 쌍 중복 확인 =====
part1 loaded, rows=503494
part2 loaded, rows=506463
part3 loaded, rows=1008060
part4 loaded, rows=717946

===== (text_A, text_B) 쌍 중복 확인 =====
part1 ↔ part2 pair overlap: 1672
part2 ↔ part3 pair overlap: 1720
part3 ↔ part4 pair overlap: 1197


In [12]:
print("\n===== [3] negative 샘플 예시 =====")

for i, path in FILES.items():
    df = pd.read_csv(path)
    if 0 in df["label"].values:
        print(f"\n--- part{i} negative samples ---")
        display(df[df["label"] == 0][["text_A", "text_B"]].head(5))



===== [3] negative 샘플 예시 =====

--- part3 negative samples ---


,text_A,text_B
0,Inclusion Criteria:\n\n* PICU monitoring for t...,Increased serum beta-hexosaminidase
2,Key Inclusion Criteria:\n\n* Chronic low back ...,Vitiligo
6,Inclusion Criteria:\n\n* Acne Vulgaris\n* Over...,Increased serum pyruvate
8,Inclusion Criteria:\n\n\- At least two of the ...,Abnormal hair pattern
10,Inclusion criteria:\n\n* Pathologically (histo...,Upper limb postural tremor



--- part4 negative samples ---


,text_A,text_B
0,Inclusion Criteria:\n\n.diagnosed as acute cor...,Abnormal T cell morphology
1,Inclusion Criteria:\n\n* Age ≥18 years;\n* Bei...,Wide nasal base
2,Inclusion Criteria:\n\n1. Age \>= 20 years\n2....,Clasp-knife sign
5,Inclusion Criteria:\n\n* Diagnosis of OCD\n* T...,Abnormality of the clavicle
7,Inclusion Criteria:\n\n1. Kidney transplant re...,Fractures of the long bones


In [11]:
print("\n===== [2] label 분포 확인 =====")
label_summary = {}

for i, path in FILES.items():
    df = pd.read_csv(path)
    counts = df["label"].value_counts()
    label_summary[i] = counts
    print(f"\npart{i} label counts:")
    print(counts)



===== [2] label 분포 확인 =====

part1 label counts:
label
1    503494
Name: count, dtype: int64

part2 label counts:
label
1    506463
Name: count, dtype: int64

part3 label counts:
label
0    504030
1    504030
Name: count, dtype: int64

part4 label counts:
label
0    358973
1    358973
Name: count, dtype: int64


In [10]:
print("===== [1] 컬럼 구조 확인 =====")
for i, path in FILES.items():
    df = pd.read_csv(path, nrows=5)
    print(f"part{i} columns → {list(df.columns)}")


===== [1] 컬럼 구조 확인 =====
part1 columns → ['text_A', 'text_B', 'label']
part2 columns → ['text_A', 'text_B', 'label']
part3 columns → ['text_A', 'text_B', 'label']
part4 columns → ['text_A', 'text_B', 'label']


In [9]:
import pandas as pd
from collections import Counter

BASE_PATH = "/content/drive/MyDrive/medical/Capstone"
FILES = {
    1: f"{BASE_PATH}/final_training_data_regex_part1.csv",
    2: f"{BASE_PATH}/final_training_data_regex_part2.csv",
    3: f"{BASE_PATH}/final_training_data_regex_part3.csv",
    4: f"{BASE_PATH}/final_training_data_regex_part4.csv",
}


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [1]:
!pip install -q transformers datasets scikit-learn accelerate
